# Bacteroides Codon Optimization Tool — Dev Notebook

Exploration/testing only. Real logic lives in `.py` module files; this notebook imports and exercises it.

Run the setup cell below first (autoreload picks up edits to the `.py` files without restarting the kernel).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '.')  # so `core`, `constraints`, `repair`, `data` import cleanly

## Small test sequences

Kept short and synthetic for fast iteration and readable cell output. Do not paste full-length genes here.

In [ ]:
# Met - Leu - Leu - Ser - STOP
test_cds = 'ATGCTATTATCATAA'
test_utr = 'AGGAGGTAAAAA'  # placeholder, not a validated Shine-Dalgarno sequence

## 1. Codon table loading (`core/codon_table.py`)

In [ ]:
from core.codon_table import load_species, available_species

print("Available species:", list(available_species().keys()))

ct = load_species('thetaiotaomicron_VPI-5482')
print(ct.display_name, '|', len(ct.codon_usage), 'codons loaded')
print('Best codon for L:', ct.best_codon('L'))
print('L codons by usage:', ct.codons_by_usage('L'))

## 2a. Full codon optimization (`core/assign_optimize.py`)

In [ ]:
from core.assign_optimize import optimize_sequence, optimize_report

optimized = optimize_sequence(test_cds, ct)
print('Input : ', test_cds)
print('Output: ', optimized)

for row in optimize_report(test_cds, ct):
    print(row)

# 2b. Codon harmonization (self-referential)

Cell  1 - sanity check the fixed-family ranking on a trivial usage dictionary

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.getcwd()  # confirm still project root

In [ ]:
from core.codon_table import load_species, source_codon_rank
from core.assign_harmonize import harmonize
from core.assign_optimize import optimize_sequence, optimize_report

Available species

In [ ]:
print(available_species().keys())

Cell 2 load target table and a short synthetic CDS

In [ ]:
from core.codon_table import load_species, available_species, source_codon_rank
from core.assign_harmonize import harmonize
from core.assign_optimize import optimize_sequence, optimize_report

# Small synthetic CDS, deliberately biasing one amino acid's codon choice
target_table = load_species("thetaiotaomicron_VPI-5482")  # use the real key from above
test_cds = "ATGCTGCTGCTCTTAGGTGGTGGCTAA"

Cell 3 inspect source rank computation in isolation

In [ ]:
rank_map = source_codon_rank(test_cds, target_table.aa_to_codons)
# print only the ranks for codons actually present in the test CDS, not all 64
present = set(test_cds[i:i+3] for i in range(0, len(test_cds)-2, 3))
for codon in present:
    print(codon, "-> rank", rank_map[codon])

Cell 4 run harmonize() end-to-end, eyeball per-position mapping

In [ ]:
seq, meta = harmonize(test_cds, target_table)
for m in meta:
    print(f"{m['source_codon']} (rank {m['source_rank']}/{m['family_size']}) -> {m['assigned_codon']}")

cell 5 round trip length/frame checks

In [ ]:
assert len(seq) == len(test_cds), "length changed"
assert len(seq) % 3 == 0, "frame broken"
print("OK:", len(seq), "nt")

Cell 6 low-rank stress test 

In [ ]:
# force a codon to a deliberately low rank and confirm it maps to the
# correspondingly low-ranked target codon, not the top one
low_rank_cds = "ATG" + "CTA"*1 + "CTG"*4 + "TAA"  # CTA appears once among 5 Leu codons -> low rank
seq2, meta2 = harmonize(low_rank_cds, target_table)
print(meta2[1])  # the CTA position

Cell 7 full-optimization vs. harmonization diff (visual gut-check, no raw seq dump)

In [ ]:
opt_seq = optimize_sequence(test_cds, target_table)
opt_report = optimize_report(test_cds, target_table)

for h, o in zip(meta, opt_report):
    marker = " <-- DIFFERS" if h["assigned_codon"] != o["new_codon"] else ""
    print(f"pos {o['position']}: harmonize={h['assigned_codon']}  optimize={o['new_codon']}{marker}")

# 2c. Codon harmonization (use imported table)

Cell 1 imports

In [ ]:
from core.codon_table import load_species, load_source_table_from_file, available_species
from core.assign_harmonize import harmonize

Cell 2 load the source-organism file

In [ ]:
import warnings

source_table = None
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    source_table = load_source_table_from_file(
    # Download a Kazusa-format table for your gene's source organism and point this at it
    "data/source_tables/Escherichia_coli_K12.txt",
    display_name="Escherichia coli K12"
)
    for w in caught:
        print(f"WARNING: {w.message}")

print(source_table.display_name, "|", len(source_table.codon_usage), "codons loaded")

Cell 3 sanity check the loaded table

In [ ]:
# Quick eyeball check: does the source table's top codon per amino acid look sane?
print("Best codon for L (Leucine):", source_table.best_codon("L"))
print("L codons by usage:", source_table.codons_by_usage("L"))

Cell 4 load target table and your test CDS

In [ ]:
target_table = load_species("thetaiotaomicron_VPI-5482")  # or whichever key you're using
test_cds = "ATGCTGCTGCTCTTAGGTGGTGGCTAA"  # same synthetic CDS as before, for continuity

Cell 5 run harmonize() with the source_table argument

In [ ]:
seq, meta = harmonize(test_cds, target_table, source_table=source_table)

for m in meta:
    print(f"{m['source_codon']} (rank {m['source_rank']}/{m['family_size']}, "
          f"method={m['source_rank_method']}) -> {m['assigned_codon']}")

Cell 6 compare against the self-referential fallback on the same CDS

In [ ]:
seq_self, meta_self = harmonize(test_cds, target_table)  # source_table=None, fallback path

for m_org, m_self in zip(meta, meta_self):
    marker = " <-- DIFFERS" if m_org["assigned_codon"] != m_self["assigned_codon"] else ""
    print(f"{m_org['source_codon']}: organism_table={m_org['assigned_codon']} "
          f"vs self_referential={m_self['assigned_codon']}{marker}")

In [ ]:
test_cds_ecoli = "ATG" + "CTA"*4 + "GGT"*2 + "TAA"

seq_org, meta_org = harmonize(test_cds_ecoli, target_table, source_table=source_table)
seq_self, meta_self2 = harmonize(test_cds_ecoli, target_table)

for m_org, m_self in zip(meta_org, meta_self2):
    marker = " <-- DIFFERS" if m_org["assigned_codon"] != m_self["assigned_codon"] else ""
    print(f"{m_org['source_codon']}: organism_table(rank {m_org['source_rank']})={m_org['assigned_codon']}  "
          f"self_referential(rank {m_self['source_rank']})={m_self['assigned_codon']}{marker}")

Cell 7 structural sanity checks

In [ ]:
assert len(seq) == len(test_cds)
assert len(seq) % 3 == 0
print("OK:", len(seq), "nt")

## 3. Constraint: 5' initiation folding (`constraints/folding.py`)

Initiate Conda environment before activating

In [ ]:
from constraints.folding import check_5prime_folding

# A synthetic 5' region with an obvious hairpin: a 10bp GC-rich stem
# (self-complementary, so it folds into a strong stem-loop) around a
# 5nt loop, right after the start codon.
cds_hairpin = "ATG" + "GCGCGCGCGC" + "AAAAA" + "GCGCGCGCGC" + "TAA"

# A flat control with essentially no self-complementarity to fold into.
cds_flat = "ATG" + "AAAAAAAAAAAAAAAAAAAAAAAAAAAAA" + "TAA"

result_hairpin = check_5prime_folding(cds_hairpin, utr=None)
result_flat = check_5prime_folding(cds_flat, utr=None)

print("Hairpin case:")
print(f"  structure : {result_hairpin.window_structure}")
print(f"  mfe       : {result_hairpin.mfe:.2f}")
print(f"  null mean : {result_hairpin.null_mean:.2f} +/- {result_hairpin.null_std:.2f}")
print(f"  z-score   : {result_hairpin.z_score:.2f}")
print(f"  passed    : {result_hairpin.passed}")

print("\nFlat case:")
print(f"  structure : {result_flat.window_structure}")
print(f"  mfe       : {result_flat.mfe:.2f}")
print(f"  null mean : {result_flat.null_mean:.2f} +/- {result_flat.null_std:.2f}")
print(f"  z-score   : {result_flat.z_score:.2f}")
print(f"  passed    : {result_flat.passed}")

## 4. Constraint: repeats (`constraints/repeats.py`)

In [ ]:
from constraints.repeats import check_repeats, reverse_complement

# clean, no repeats of length >= 11
test_clean = "ATGAAATTTAAATTTAAATAAAGGGCCCTTTAGA"
print(len(check_repeats(test_clean)))  # expect 0

# exact DIRECT repeat: same 12bp block appears twice, far apart
block = "ATGCATGCATGC"  # 12bp
test_direct = block + "TTTTTTTTTTTTTTTT" + block
print(check_repeats(test_direct, max_mismatches=0))  # expect >=1 direct_exact hit

# exact INVERTED repeat: a 12bp block, then its reverse complement later
test_inverted = block + "TTTTTTTTTTTTTTTT" + reverse_complement(block)
print(check_repeats(test_inverted, max_mismatches=0))  # expect >=1 inverted_exact hit

# fuzzy direct repeat: same block but with 1 mismatch introduced
block_mut = "ATGCATGCATTC"  # one base changed near the end (C->T at position 10... adjust as needed)
test_fuzzy = block + "TTTTTTTTTTTTTTTT" + block_mut
exact_hits = check_repeats(test_fuzzy, max_mismatches=0)
fuzzy_hits = check_repeats(test_fuzzy, max_mismatches=2)
print("exact:", len(exact_hits), "| fuzzy (incl. exact tier):", len(fuzzy_hits))

## 5. Constraint: GC content (`constraints/gc_content.py`)

In [ ]:
from constraints.gc_content import check_gc_content, gc_fraction

test_seq_ok = "ATGAAATTTAAATTTAAATAA"
print(check_gc_content(test_seq_ok))  # expect []

test_seq_gc_rich = "ATG" + "GCG"*10 + "TAA"
print(gc_fraction(test_seq_gc_rich))
print(check_gc_content(test_seq_gc_rich))  # expect one GCViolation

In [ ]:
test_seq_local_hotspot = "ATT"*20 + "GCG"*17 + "AAT"*20
print(gc_fraction(test_seq_local_hotspot))            # expect this to be well under 0.70
print(len(check_gc_content(test_seq_local_hotspot)))   # expect > 0 despite global being clean

## 6. Constraint: restriction sites (`constraints/restriction.py`)

In [ ]:
from constraints.restriction import check_restriction_sites, reverse_complement

# should find nothing
test_clean = "ATGAAATTTAAATTTAAATAA"
print(check_restriction_sites(test_clean))  # expect []

# forward-strand EcoRI site embedded
test_ecori = "ATG" + "GAATTC" + "TAA"
print(check_restriction_sites(test_ecori))  # expect one EcoRI hit

# BsaI site on the reverse strand only (i.e. reverse complement of GGTCTC = GAGACC present, not GGTCTC itself)
test_bsai_reverse = "ATG" + reverse_complement("GGTCTC") + "TAA"
print(check_restriction_sites(test_bsai_reverse))  # expect one BsaI hit, strand="reverse"

## 7. Repair loop (`repair/repair_engine.py`)

Quick sanity checks

In [ ]:
from constraints.restriction import check_restriction_sites

# EcoRI site placed at a known position, 0-indexed
site = "GAATTC"
test_seq = "AAAA" + site + "TTTT"   # site should sit at seq[4:10]

hits = check_restriction_sites(test_seq)
for h in hits:
    print(h.enzyme, h.start, h.end, h.strand, h.matched_seq)
    print("  seq[start:end] =", test_seq[h.start:h.end])
    print("  matches?", test_seq[h.start:h.end] == h.matched_seq)

In [ ]:
from constraints.repeats import check_repeats

# two identical 11-mers, far enough apart not to overlap, at known positions
repeat_unit = "ACGTACGTACG"  # len 11 == default min_len
test_seq = repeat_unit + "TTTTTTTTTT" + repeat_unit  # copy 2 starts at 11+10 = 21

hits = check_repeats(test_seq)
for h in hits:
    print(h.kind, h.start1, h.end1, h.start2, h.end2, h.mismatches)
    print("  copy1:", test_seq[h.start1:h.end1])
    print("  copy2:", test_seq[h.start2:h.end2])
    print("  copy1 == copy2?", test_seq[h.start1:h.end1] == test_seq[h.start2:h.end2])

In [ ]:
from constraints.gc_content import check_gc_content, gc_fraction

test_seq = "AT" * 5 + "GC" * 20 + "AT" * 5   # a clearly high-GC stretch in the middle

hits = check_gc_content(test_seq, ceiling=0.5)
for h in hits:
    recomputed = gc_fraction(test_seq[h.start:h.end])
    print(h.start, h.end, h.gc_fraction, "recomputed:", recomputed, "match?", abs(h.gc_fraction - recomputed) < 1e-9)

demo cell for the repair loop, mirroring existing section structure

In [ ]:
import inspect
import constraints.repeats as repeats
import constraints.gc_content as gc_content
import constraints.restriction as restriction

for mod, fn_name, result_name in [
    (repeats, "check_repeats", "RepeatViolation"),
    (gc_content, "check_gc_content", "GCViolation"),
    (restriction, "check_restriction_sites", "RestrictionViolation"),
]:
    fn = getattr(mod, fn_name)
    result_cls = getattr(mod, result_name)
    print(mod.__name__)
    print(" signature:", inspect.signature(fn))
    print(" fields:", list(result_cls.__dataclass_fields__.keys()))
    print()

In [ ]:
from core.codon_table import load_species
from repair.repair_engine import repair_sequence

table = load_species('thetaiotaomicron_VPI-5482')  # or whichever key your metadata.json uses

# short synthetic CDS, per your convention -- not a full-length gene
test_cds = "ATGTTATTAAAAGAATAT"  # Met-Leu-Leu-Lys-Glu-Tyr, deliberately has adjacent TTA/TTA

result = repair_sequence(test_cds, table, mode="optimize")

print("converged:", result.converged, " passes:", result.passes_run)
print("substitutions made:", len(result.actions))
for a in result.actions:
    print(f"  pass {a.pass_num}: codon {a.codon_index} {a.old_codon}->{a.new_codon} (fixed {a.fixed_violation})")

Harmonize-mode cell

In [ ]:
from core.assign_harmonize import harmonize

harmonized_seq, harmonize_metadata = harmonize(test_cds, table)
result_h = repair_sequence(
    harmonized_seq, table, mode="harmonize",
    harmonize_metadata=harmonize_metadata,
)
print("converged:", result_h.converged)
for a in result_h.actions:
    print(f"  codon {a.codon_index}: rank {a.rank_before}->{a.rank_after}")

## 8. Diagnostics report (`report.py`)

Diagnostics report cell

In [ ]:
from report import build_report

rpt = build_report("bacteroides_thetaiotaomicron", "optimize", result.sequence, result)
print(rpt.to_text())